In [1]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [2]:
from pathlib import Path

csv_path = (
    Path("data/adult.csv") if Path("data/adult.csv").exists()
    else Path("../data/adult.csv") if Path("../data/adult.csv").exists()
    else Path("adult.csv")
)
df = pd.read_csv(csv_path, index_col=0)

# '?' is used as a missing-value marker in the categorical columns.
df = df.replace(r"^\s*\?\s*$", np.nan, regex=True).dropna().copy()

# Check that the two salary representations agree with the $50K threshold.
salary_valid = df["salary"].eq("<=50K") | df["salary"].eq(">50K")
salary_consistent = df["salary"].eq(">50K").eq(df["salary K$"].gt(50))
assert salary_valid.all() and salary_consistent.all(), "Inconsistent salary data"

print(f"Clean rows: {len(df):,}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Salary values consistent: {salary_consistent.all()}")


Clean rows: 30,162
Missing values: 0
Salary values consistent: True


# Task 1
Print the count of men and women in the dataset.

In [3]:
print(df["sex"].value_counts())

sex
Male      20380
Female     9782
Name: count, dtype: int64


# Task 2
Find the average age of men in dataset

In [4]:
print(f"Average age of men: {df.loc[df['sex'].eq('Male'), 'age'].mean():.2f}")

Average age of men: 39.18


# Task 3
Get the percentage of people from Poland (native-country)

In [5]:
poland_share = df["native-country"].eq("Poland").mean() * 100
print(f"People from Poland: {poland_share:.2f}%")

People from Poland: 0.19%


# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [6]:
age_by_salary = df.groupby("salary")["age"].agg(["mean", "std"])
print(age_by_salary)

            mean        std
salary                     
<=50K   36.60806  13.464631
>50K    43.95911  10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [7]:
higher_education = df["education"].str.fullmatch(
    r"Bachelors|Prof-school|Assoc-acdm|Assoc-voc|Masters|Doctorate"
)
high_income_without_degree = df.loc[
    df["salary"].eq(">50K") & ~higher_education
]
print(f"People earning >50K without higher education: {len(high_income_without_degree)}")
print(f"Exist: {not high_income_without_degree.empty}")

People earning >50K without higher education: 3178
Exist: True


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [8]:
print(df.groupby("education")["age"].describe())

               count       mean        std   min   25%   50%   75%   max
education                                                               
10th           820.0  37.897561  16.225795  17.0  23.0  36.0  52.0  90.0
11th          1048.0  32.363550  15.089307  17.0  18.0  28.5  43.0  90.0
12th           377.0  32.013263  14.373710  17.0  19.0  28.0  41.0  79.0
1st-4th        151.0  44.622517  14.929051  19.0  33.0  44.0  56.0  81.0
5th-6th        288.0  41.649306  14.754622  17.0  28.0  41.0  53.0  82.0
7th-8th        557.0  47.631957  15.737479  17.0  34.0  49.0  60.0  90.0
9th            455.0  40.303297  15.335754  17.0  28.0  38.0  53.0  90.0
Assoc-acdm    1008.0  37.286706  10.509755  19.0  29.0  36.0  44.0  90.0
Assoc-voc     1307.0  38.246366  11.181253  19.0  30.0  37.0  45.0  84.0
Bachelors     5044.0  38.641554  11.577566  19.0  29.0  37.0  46.0  90.0
Doctorate      375.0  47.130667  11.471727  24.0  39.0  47.0  54.0  80.0
HS-grad       9840.0  38.640955  13.067730  17.0  2

# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [9]:
men = df.loc[df["sex"].eq("Male")].copy()
men["married"] = men["marital-status"].str.startswith("Married")

salary_counts = pd.crosstab(men["married"], men["salary"])
salary_percentages = pd.crosstab(
    men["married"], men["salary"], normalize="index"
).mul(100).round(2)

print("Salary counts (False = unmarried, True = married):")
print(salary_counts)
print("\nWithin-group percentages:")
print(salary_percentages)
print("\nMean salary, $K:")
print(men.groupby("married")["salary K$"].mean().round(2))

Salary counts (False = unmarried, True = married):
salary   <=50K  >50K
married             
False     6932   673
True      7052  5723

Within-group percentages:
salary   <=50K   >50K
married              
False    91.15   8.85
True     55.20  44.80

Mean salary, $K:
married
False     46.60
True     107.49
Name: salary K$, dtype: float64


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [10]:
max_hours = df["hours-per-week"].max()
people_at_max = df["hours-per-week"].eq(max_hours).sum()
print(f"Maximum hours per week: {max_hours}")
print(f"Number of people working this many hours: {people_at_max}")

Maximum hours per week: 99
Number of people working this many hours: 78


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [12]:
# Pearson correlations for real numeric variables (salary K$ is an actual numeric column).
numeric_correlations = df.select_dtypes(include="number").corr()
print("Numeric variable correlations:")
print(numeric_correlations.round(3))

# Binary indicators allow meaningful comparison of categorical values with >50K salary.
# Avoid ordinal-encoding nominal categories, which would create arbitrary correlations.
high_income = df["salary"].eq(">50K").astype("int8")
category_indicators = pd.get_dummies(
    df.select_dtypes(include=["object", "category"]).drop(columns="salary"),
    dtype="int8"
)
category_salary_correlation = category_indicators.corrwith(high_income).dropna()

print("\nTop 10 positive associations with salary >50K:")
print(category_salary_correlation.nlargest(10).round(3))
print("\nTop 10 negative associations with salary >50K:")
print(category_salary_correlation.nsmallest(10).round(3))
print("\nNotes: correlation measures association, not causation.")
print("salary K$ directly determines the salary category, so its strong link is expected.")

Numeric variable correlations:
                  age  hours-per-week  salary K$
age             1.000           0.102      0.208
hours-per-week  0.102           1.000      0.196
salary K$       0.208           0.196      1.000


/tmp/ipykernel_44509/341410068.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=["object", "category"]).drop(columns="salary"),



Top 10 positive associations with salary >50K:
marital-status_Married-civ-spouse    0.445
relationship_Husband                 0.401
sex_Male                             0.217
occupation_Exec-managerial           0.213
occupation_Prof-specialty            0.181
education_Bachelors                  0.179
education_Masters                    0.174
education_Prof-school                0.156
workclass_Self-emp-inc               0.138
education_Doctorate                  0.129
dtype: float64

Top 10 negative associations with salary >50K:
marital-status_Never-married   -0.320
relationship_Own-child         -0.226
sex_Female                     -0.217
relationship_Not-in-family     -0.193
occupation_Other-service       -0.166
relationship_Unmarried         -0.146
education_HS-grad              -0.136
marital-status_Divorced        -0.132
workclass_Private              -0.117
occupation_Adm-clerical        -0.100
dtype: float64

Notes: correlation measures association, not causation.
salary 